In [1]:
from ta.volatility import AverageTrueRange

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

from backtesting import Strategy
from backtesting import Backtest


from datetime import timedelta

import sys
sys.path.append('../utils')
import Util
import importlib
importlib.reload(Util)

from enum import Enum
import time

from random import randint
import random

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

In [2]:
pair = 'CADCHF'

df_d1_raw = pd.read_csv(f"../Trading Data/{pair}_D1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_h4_raw = pd.read_csv(f"../Trading Data/{pair}_H4.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_m15_raw = pd.read_csv(f"../Trading Data/{pair}_M15.csv",sep=',', parse_dates=['Date'], index_col='Date')

print(df_d1_raw)
print(df_h4_raw)
print(df_m15_raw)

               Open     High      Low    Close  Tickvol
Date                                                   
2010-09-23  0.95790  0.95845  0.94550  0.95260    81300
2010-09-24  0.95260  0.96180  0.94780  0.95980   103852
2010-09-27  0.96055  0.96420  0.95510  0.95655   115385
2010-09-28  0.95655  0.95765  0.94310  0.94775   126433
2010-09-29  0.94765  0.95425  0.94400  0.94620   115335
...             ...      ...      ...      ...      ...
2026-10-05  0.58075  0.58381  0.58046  0.58250    48951
2026-10-06  0.58249  0.58551  0.58186  0.58538    42990
2026-10-07  0.58540  0.58631  0.58206  0.58423    50257
2026-10-08  0.58424  0.58563  0.58294  0.58443    66525
2026-10-09  0.58444  0.58472  0.58121  0.58234    34325

[4185 rows x 5 columns]
                        Open     High      Low    Close  Tickvol
Date                                                            
2010-09-23 08:00:00  0.95790  0.95845  0.95285  0.95370    19153
2010-09-23 12:00:00  0.95370  0.95455  0.94550  0.95

In [3]:
structure_d1_window = 2
structure_m15_window = 2

date_format = '%Y-%m-%d %H:%M'
# start_date = '2023-1-1 00:00'
# end_date = '2024-3-29 00:00'
start_date = datetime.strptime('2021-1-1 00:00', date_format)
end_date = datetime.strptime('2026-9-16 00:00', date_format)
# start_date = '2022-01-1 00:00'

timezone = "Europe/London"

In [4]:
# process D1 data
df_d1 = df_d1_raw.loc[start_date - timedelta(days=30) : end_date]

df_d1 = Util.implement_market_structure_states_fixed_pivots(df_d1, n=structure_d1_window)
df_d1["state_changed"] = df_d1["market_state"] != df_d1[
    "market_state"
].shift(1)
print(df_d1[df_d1["state_changed"]][["Close", "market_state"]])
df_d1 = df_d1.loc[start_date : end_date]

              Close market_state
Date                            
2020-12-02  0.69255          NaN
2020-12-03  0.69245          NaN
2020-12-04  0.69728          NaN
2020-12-07  0.69550          NaN
2020-12-08  0.69335          NaN
...             ...          ...
2026-06-05  0.57084      BULLISH
2026-07-02  0.56666      BEARISH
2026-07-13  0.57560      BULLISH
2026-07-30  0.57487      BEARISH
2026-08-11  0.58244      BULLISH

[108 rows x 2 columns]


In [5]:
# process h4 data
df_h4 = df_h4_raw.loc[start_date - timedelta(days=20) : end_date]
df_h4 = Util.implement_market_structure_states_fixed_pivots(df_h4, 5)
df_h4 = df_h4.loc[start_date : end_date]

h4_d1_direction_list = []
for i in range(len(df_d1)):
    row = df_d1.iloc[i]
    start_date_daily = row.name
    end_date_daily = start_date_daily + timedelta(hours=23, minutes=59)
    df_by_day = df_h4.loc[f'{start_date_daily}' : f'{end_date_daily}']

    for index2, row2 in df_by_day.iterrows():
        h4_d1_direction_list.append(df_d1.iloc[i-1]['market_state'])

df_h4['market_state_d1'] = h4_d1_direction_list

indicator_atr_14 = AverageTrueRange(df_h4['High'], df_h4['Low'], df_h4['Close'], window=14)
df_h4['ATR'] = indicator_atr_14._atr

df_h4

,Open,High,Low,Close,Tickvol,is_swing_high,is_swing_low,market_state,market_state_d1,ATR
Date,,,,,,,,,,
2021-01-04 00:00:00,0.69378,0.69573,0.69310,0.69555,12960,False,False,BULLISH,BULLISH,0.000000
2021-01-04 04:00:00,0.69553,0.69625,0.69513,0.69527,9826,True,False,BULLISH,BULLISH,0.000000
2021-01-04 08:00:00,0.69525,0.69580,0.69311,0.69377,20631,False,False,BULLISH,BULLISH,0.000000
2021-01-04 12:00:00,0.69377,0.69405,0.69151,0.69293,23712,False,False,BULLISH,BULLISH,0.000000
2021-01-04 16:00:00,0.69296,0.69395,0.68858,0.68984,32860,False,False,BULLISH,BULLISH,0.000000
...,...,...,...,...,...,...,...,...,...,...
2026-09-15 08:00:00,0.58789,0.58839,0.58749,0.58791,6852,False,False,BULLISH,BULLISH,0.001281
2026-09-15 12:00:00,0.58792,0.58904,0.58735,0.58878,9087,False,False,BULLISH,BULLISH,0.001310
2026-09-15 16:00:00,0.58879,0.58880,0.58812,0.58812,4546,False,False,BULLISH,BULLISH,0.001265


In [6]:
# process m15 data

df_m15 = df_m15_raw.loc[start_date - timedelta(hours=3): end_date]
df_m15 = Util.implement_market_structure_states_fixed_pivots(df_m15, n=structure_m15_window)

m15_daily_direction_list = []

index_list = []

indicator_atr_m15 = AverageTrueRange(df_m15['High'], df_m15['Low'], df_m15['Close'], window=14)
df_m15['ATR'] = indicator_atr_m15._atr

df_m15 = df_m15.loc[start_date : end_date]
for i in range(len(df_h4)):
    row = df_h4.iloc[i]
    start_date_h4 = row.name
    end_date_h4 = start_date_h4 + timedelta(hours=3, minutes=59)
    df_by_day = df_m15.loc[f'{start_date_h4}' : f'{end_date_h4}']

    for index2, row2 in df_by_day.iterrows():
        m15_daily_direction_list.append(df_h4.iloc[i]['market_state_d1'])

        # ajust to daylight saving time
        dst = pd.Timestamp(index2, tz=timezone).dst()
        if dst == timedelta(0):
            index2 = index2 - timedelta(hours=1)
            index_list.append(index2)
        else:
            index_list.append(index2)

df_m15['market_state_d1'] = m15_daily_direction_list

df_m15.index = index_list

In [26]:
# add some sl tp management

class RandomStrat(Strategy):
    mysize = 0.5
    slcoef = 2
    TPcoef = 2
    risk_percent = 0.01
    trades_list = None
    stoploss_window = 150

    def is_signal(self, date):
        for i in range(0, len(self.trades_list)):
            if self.trades_list[i] == date:
                return True
        return False

    def get_highest_high(self):
        return self.data.High[len(self.data.High) - self.stoploss_window : len(self.data.High)].max()
    
    def get_lowest_low(self):
        return self.data.Low[len(self.data.Low) - self.stoploss_window : len(self.data.Low)].min()
    
    def init(self):
        super().init()

    def get_sl(self):
        return round(random.uniform(self.data['ATR'][-1]*1,self.data['ATR'][-1]*2.5), 5)

    def next(self):
        super().next()
        if self.is_signal(self.data.index[-1]) and len(self.trades) == 0 and len(self.closed_trades) < 80:
            direction = randint(1,2)
            if direction == 1:
                planned_entry_price = self.data.Close[-1]
                sl = abs(planned_entry_price - self.get_lowest_low())
                if sl == 0:
                    sl = self.get_sl()
                tp = self.TPcoef * sl

                sl1 = planned_entry_price - sl
                tp1 = planned_entry_price + tp
                position_size = int((self.risk_percent * self.equity) / sl)

                self.buy(sl=sl1, tp=tp1, size=position_size)
            else:
                planned_entry_price = self.data.Close[-1]
                sl = abs(planned_entry_price - self.get_highest_high())
                if sl == 0:
                    sl = self.get_sl()
                tp = self.TPcoef * sl

                sl1 = planned_entry_price + sl
                tp1 = planned_entry_price - tp
                position_size = int((self.risk_percent * self.equity) / sl)

                self.sell(sl=sl1, tp=tp1, size=position_size)

In [38]:
trades_list = []
while len(trades_list) != 110:
    random_index = randint(0, len(df_m15)-1)
    candle = df_m15.iloc[random_index]
    # exclu Saturday and Sunday
    if candle.name.weekday() < 5:
        if 9 <= candle.name.hour and candle.name.hour <= 19:
            trades_list.append(candle.name)
trades_list.sort()

bt = Backtest(df_m15, RandomStrat, cash=10000, margin=0.005, finalize_trades=True) #0.0002
stats = bt.run(trades_list=trades_list)

print(stats)

bt.plot()

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=85949: Broker canceled the order due to insufficient margin (equity=9084.39, margin_available=9084.39).
  warnings.warn(


Start                     2021-01-03 23:00:00
End                       2026-09-16 00:00:00
Duration                   2081 days 01:00:00
Exposure Time [%]                    12.62709
Equity Final [$]                  10589.99185
Equity Peak [$]                   11297.07695
Return [%]                            5.89992
Buy & Hold Return [%]                -15.2726
Return (Ann.) [%]                     0.90411
Volatility (Ann.) [%]                  5.1229
CAGR [%]                              1.01119
Sharpe Ratio                          0.17648
Sortino Ratio                          0.2918
Calmar Ratio                          0.08377
Alpha [%]                             5.67231
Beta                                  -0.0149
Max. Drawdown [%]                   -10.79265
Avg. Drawdown [%]                    -0.75048
Max. Drawdown Duration     1603 days 08:45:00
Avg. Drawdown Duration       36 days 00:23:00
# Trades                                   80
Win Rate [%]                      

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\_plotting.py:141: UserWarning: Data contains too many candlesticks to plot; downsampling to '8h'. See `Backtest.plot(resample=...)`
  warnings.warn(f"Data contains too many candlesticks to plot; downsampling to {freq!r}. "


GridPlot(id='p8361', ...)

In [39]:
i = 0
expectancy_list = []
winrate_list = []

while len(expectancy_list) < 1000:
    trades_list = []
    while len(trades_list) != 110:
        random_index = randint(0, len(df_m15)-1)
        candle = df_m15.iloc[random_index -1]
        # exclu Saturday and Sunday
        if candle.name.weekday() < 5:
            if 9 <= candle.name.hour and candle.name.hour <= 19:
                trades_list.append(candle.name)
    trades_list.sort()

    bt = Backtest(df_m15, RandomStrat, cash=10000, margin=0.009, finalize_trades=True) #0.0002
    stats = bt.run(trades_list=trades_list)
    if stats['# Trades'] == 80:
        expectancy_list.append(stats['Expectancy [%]'])
        winrate_list.append(stats['Win Rate [%]'])
        print(f'{i} {len(expectancy_list)} {stats['Expectancy [%]']} Done')
    i += 1

print(expectancy_list)

0 1 -0.06368039146131044 Done


c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=23616: Broker canceled the order due to insufficient margin (equity=9498.93, margin_available=9498.93).
  warnings.warn(


1 2 -0.13213922663623381 Done
3 3 0.08947290961568641 Done


c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=26660: Broker canceled the order due to insufficient margin (equity=9306.65, margin_available=9306.65).
  warnings.warn(


4 4 -0.14285415271087468 Done
5 5 0.025471866110698682 Done
6 6 -0.0047350761878545145 Done


...

1328 997 -0.018422010719560955 Done


c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=23518: Broker canceled the order due to insufficient margin (equity=9891.82, margin_available=9891.82).
  warnings.warn(


1329 998 -0.043649638976867994 Done


c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=83374: Broker canceled the order due to insufficient margin (equity=9347.59, margin_available=9347.59).
  warnings.warn(


1330 999 0.0005909561226129234 Done
1331 1000 -0.017114315267363206 Done
[np.float64(-0.06368039146131044), np.float64(-0.13213922663623381), np.float64(0.08947290961568641), np.float64(-0.14285415271087468), np.float64(0.025471866110698682), np.float64(-0.0047350761878545145), np.float64(0.06186950868419999), np.float64(-0.08030901678876004), np.float64(-0.10990027127896762), np.float64(-0.0047951113586354455), np.float64(0.10022169737999519), np.float64(-0.1294301904554017), np.float64(0.03256751967098154), np.float64(0.05559403050464332), np.float64(-0.05481983303498986), np.float64(-0.05970589470890673), np.float64(0.08550771646471972), np.float64(-0.009731217561733119), np.float64(-0.031177891070191116), np.float64(0.041420153246602004), np.float64(-0.03661917573289181), np.float64(0.07504940513076763), np.float64(0.06285717914643052), np.float64(-0.06609820938475675), np.float64(0.045195547534772895), np.float64(-0.0713309377233122), np.float64(-0.01790272995896236), np.float64(0

In [40]:
print(len(expectancy_list))

p95 = np.percentile(expectancy_list, 95)                  # ngưỡng phân vị 95
print(f"p95 = {p95}")
# max_expectancy = max(expectancy_list_2)
print(f"max = {max(expectancy_list)}")
mean_expectancy = np.mean(expectancy_list)
print(f"mean = {mean_expectancy}")
print(f"min = {min(expectancy_list)}")
print(f"gap = {-0.02684 - mean_expectancy}")

df_result = pd.DataFrame({'expectancy' : expectancy_list})
rank = (df_result['expectancy'] <= -0.02684).mean() * 100 
print(f"rank = {rank}")
p_value = ((df_result['expectancy'] >= -0.02684).sum() + 1) / (len(df_result['expectancy']) + 1)
print(f"p_value = {p_value}")

print(len(winrate_list))

p95_winrate = np.percentile(winrate_list, 95)                  # ngưỡng phân vị 95
print(f"p95 = {p95_winrate}")
# max_expectancy = max(expectancy_list_2)
print(f"max = {max(winrate_list)}")
mean_winrate = np.mean(winrate_list)
print(f"mean = {mean_winrate}")
print(f"min = {min(winrate_list)}")
print(f"gap = {45.09804 - mean_winrate}")

1000
p95 = 0.10756528376922987
max = 0.25119338082948073
mean = -0.044033197766299825
min = -0.38739692102857826
gap = 0.017193197766299826
rank = 57.599999999999994
p_value = 0.4245754245754246
1000
p95 = 40.0
max = 47.5
mean = 31.42
min = 16.25
gap = 13.678039999999996


In [ ]:
# add some sl tp management

class RandomBiasStrat(Strategy):
    mysize = 0.5
    slcoef = 2
    TPcoef = 2
    risk_percent = 0.01
    trades_list = None
    stoploss_window = 150

    def is_signal(self, date):
        for i in range(0, len(self.trades_list)):
            if self.trades_list[i] == date:
                return True
        return False

    def get_highest_high(self):
        return self.data.High[len(self.data.High) - self.stoploss_window : len(self.data.High)].max()
    
    def get_lowest_low(self):
        return self.data.Low[len(self.data.Low) - self.stoploss_window : len(self.data.Low)].min()
    
    def init(self):
        super().init()

    def next(self):
        super().next()
        if self.is_signal(self.data.index[-1]) and len(self.trades) == 0 and len(self.closed_trades) < 63:
            if self.data['market_state_d1'][-1] == 'BEARISH':
                planned_entry_price = self.data.Close[-1]
                sl = abs(planned_entry_price - self.get_lowest_low())
                if sl == 0:
                    sl = round(random.uniform(0.3,1), 1)
                tp = self.TPcoef * sl

                sl1 = planned_entry_price - sl
                tp1 = planned_entry_price + tp
                position_size = int((self.risk_percent * self.equity) / sl)
                # max_affordable_size = int((self.equity * 0.99) / planned_entry_price)
                # print(f'{position_size} {max_affordable_size}')

                self.buy(sl=sl1, tp=tp1, size=position_size)
            else:
                planned_entry_price = self.data.Close[-1]
                sl = abs(planned_entry_price - self.get_highest_high())
                if sl == 0:
                    sl = round(random.uniform(0.3,1), 1)
                tp = self.TPcoef * sl

                sl1 = planned_entry_price + sl
                tp1 = planned_entry_price - tp
                position_size = int((self.risk_percent * self.equity) / sl)
                # max_affordable_size = int((self.equity * 0.99) / planned_entry_price)
                # print(f'{position_size} {max_affordable_size}')

                self.sell(sl=sl1, tp=tp1, size=position_size)

In [ ]:
trades_list = []
while len(trades_list) != 73:
    random_index = randint(0, len(df_m15))
    candle = df_m15.iloc[random_index]
    # exclu Saturday and Sunday
    if candle.name.weekday() < 5:
        if 9 <= candle.name.hour and candle.name.hour <= 19:
            trades_list.append(candle.name)
trades_list.sort()

bt = Backtest(df_m15, RandomBiasStrat, cash=10000, margin=0.005, finalize_trades=True) #0.0002
stats = bt.run(trades_list=trades_list)

print(stats)

bt.plot()

Start                     2021-01-03 23:00:00
End                       2026-09-16 00:00:00
Duration                   2081 days 01:00:00
Exposure Time [%]                    15.82804
Equity Final [$]                      9455.05
Equity Peak [$]                     10058.825
Return [%]                            -5.4495
Buy & Hold Return [%]                48.17505
Return (Ann.) [%]                    -0.87596
Volatility (Ann.) [%]                 4.03178
CAGR [%]                             -0.97869
Sharpe Ratio                         -0.21726
Sortino Ratio                        -0.31596
Calmar Ratio                         -0.10401
Alpha [%]                             -7.1519
Beta                                  0.03534
Max. Drawdown [%]                    -8.42151
Avg. Drawdown [%]                    -3.16061
Max. Drawdown Duration     1040 days 08:30:00
Avg. Drawdown Duration      403 days 06:45:00
# Trades                                   62
Win Rate [%]                      

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\_plotting.py:141: UserWarning: Data contains too many candlesticks to plot; downsampling to '8h'. See `Backtest.plot(resample=...)`
  warnings.warn(f"Data contains too many candlesticks to plot; downsampling to {freq!r}. "


GridPlot(id='p1261', ...)

In [ ]:
i = 0
expectancy_list = []
winrate_list = []

while len(expectancy_list) < 1000:
    trades_list = []
    while len(trades_list) != 80:
        random_index = randint(0, len(df_m15)-1)
        candle = df_m15.iloc[random_index -1]
        # exclu Saturday and Sunday
        if candle.name.weekday() < 5:
            if 9 <= candle.name.hour and candle.name.hour <= 19:
                trades_list.append(candle.name)
    trades_list.sort()

    bt = Backtest(df_m15, RandomBiasStrat, cash=10000, margin=0.009, finalize_trades=True) #0.0002
    stats = bt.run(trades_list=trades_list)
    if stats['# Trades'] == 63:
        expectancy_list.append(stats['Expectancy [%]'])
        winrate_list.append(stats['Win Rate [%]'])
        print(f'{i} {len(expectancy_list)} {stats['Expectancy [%]']} Done')
    i += 1

print(expectancy_list)

In [ ]:
print(len(expectancy_list))

p95 = np.percentile(expectancy_list, 95)                  # ngưỡng phân vị 95
print(f"p95 = {p95}")
# max_expectancy = max(expectancy_list_2)
print(f"max = {max(expectancy_list)}")
mean_expectancy = np.mean(expectancy_list)
print(f"mean = {mean_expectancy}")
print(f"min = {min(expectancy_list)}")
print(f"gap = {0.34589 - mean_expectancy}")

df_result = pd.DataFrame({'expectancy' : expectancy_list})
rank = (df_result['expectancy'] <= 0.34589).mean() * 100 
print(f"rank = {rank}")
p_value = ((df_result['expectancy'] >= 0.34589).sum() + 1) / (len(df_result['expectancy']) + 1)
print(f"p_value = {p_value}")

print(len(winrate_list))

p95_winrate = np.percentile(winrate_list, 95)                  # ngưỡng phân vị 95
print(f"p95 = {p95_winrate}")
# max_expectancy = max(expectancy_list_2)
print(f"max = {max(winrate_list)}")
mean_winrate = np.mean(winrate_list)
print(f"mean = {mean_winrate}")
print(f"min = {min(winrate_list)}")
print(f"gap = {45.09804 - mean_winrate}")